In [ ]:
import re
import pandas as pd

# 1. Leer el archivo original de 2017-2016 como texto plano
with open('llegadas_2017_2016_clean.csv', 'r', encoding='utf-8-sig', errors='ignore') as f:
    lineas = f.readlines()

lineas_datos = [l.strip() for l in lineas if l.strip() != ""]
if "RO" in lineas_datos[0]:
    lineas_datos = lineas_datos[1:]

# 2. ORDEN INVERTIDO DE LAS COLUMNAS (De más nuevo a más viejo: Dic 2017 -> Ene 2016)
meses_columnas = [
    "2017-12", "2017-11", "2017-10", "2017-09", "2017-08", "2017-07",
    "2017-06", "2017-05", "2017-04", "2017-03", "2017-02", "2017-01",
    "2016-12", "2016-11", "2016-10", "2016-09", "2016-08", "2016-07",
    "2016-06", "2016-05", "2016-04", "2016-03", "2016-02", "2016-01"
]

matriz_paises = {}

# 3. Extraer la serie temporal de cada país
for linea in lineas_datos[1:]:
    if "Pasajeros Totales" in linea:
        continue

    paises_encontrados = re.findall(r'[A-ZÁÉÍÓÚÑüÜ]{3,}(?:\s+[A-ZÁÉÍÓÚÑüÜ]+)*', linea)
    if not paises_encontrados:
        continue
    pais = paises_encontrados[0].strip()

    numeros = re.findall(r'"([\d\.]+)"', linea)
    if not numeros:
        numeros = re.findall(r'\b\d+(?:\.\d+)*\b', linea)

    valores_meses = []
    for num in numeros:
        num_limpio = num.replace('.', '')
        if num_limpio.isdigit():
            valores_meses.append(int(num_limpio))

    matriz_paises[pais] = valores_meses

# 4. Construir la estructura temporal mes a mes
paises_top = {'ESPAÑA', 'SPAIN', 'FRANCIA', 'FRANCE', 'ITALIA', 'ITALY',
              'REINO UNIDO', 'UNITED KINGDOM', 'UK', 'PAISES BAJOS', 'HOLANDA',
              'NETHERLANDS', 'ALEMANIA', 'GERMANY'}

datos_historicos_finales = []

for idx_mes, mes_nombre in enumerate(meses_columnas):

    def extraer_valor_mes(nombre_pais):
        for k, lista_valores in matriz_paises.items():
            if nombre_pais in k and idx_mes < len(lista_valores):
                return lista_valores[idx_mes]
        return 0

    nac = extraer_valor_mes('ESPAÑA') or extraer_valor_mes('SPAIN')
    fr = extraer_valor_mes('FRANCIA') or extraer_valor_mes('FRANCE')
    it = extraer_valor_mes('ITALIA') or extraer_valor_mes('ITALY')
    uk = extraer_valor_mes('REINO UNIDO') or extraer_valor_mes('UNITED KINGDOM') or extraer_valor_mes('UK')
    nl = extraer_valor_mes('PAISES BAJOS') or extraer_valor_mes('HOLANDA') or extraer_valor_mes('NETHERLANDS')
    de = extraer_valor_mes('ALEMANIA') or extraer_valor_mes('GERMANY')

    resto_del_mundo = 0
    for k, lista_valores in matriz_paises.items():
        if not any(p in k for p in paises_top) and idx_mes < len(lista_valores):
            resto_del_mundo += lista_valores[idx_mes]

    internacionales = 0
    for k, lista_valores in matriz_paises.items():
        if 'ESPAÑA' not in k and 'SPAIN' not in k and idx_mes < len(lista_valores):
            internacionales += lista_valores[idx_mes]

    total_aeropuerto = nac + internacionales

    datos_historicos_finales.append({
        'Mes': mes_nombre,
        'Volumen total llegadas aeropuerto': total_aeropuerto,
        'Volumen llegadas Nacionales': nac,
        'Volumen llegadas Internacionales': internacionales,
        'Volumen llegadas Francia': fr,
        'Volumen llegadas Italia': it,
        'Volumen llegadas Reino Unido': uk,
        'Volumen llegadas Países Bajos': nl,
        'Volumen llegadas Alemania': de,
        'Volumen llegadas Resto del Mundo': resto_del_mundo
    })

# 5. Convertir a DataFrame, ordenar cronológicamente (Ene 2016 -> Dic 2017) y guardar
df_temporal = pd.DataFrame(datos_historicos_finales)
df_temporal = df_temporal.sort_values(by='Mes').reset_index(drop=True)

# Guardar
df_temporal.to_csv('2017-2016.csv', index=False, encoding='utf-8-sig')

df_temporal

¡Solucionado! Tu archivo corregido ha sido guardado como: '2017-2016.csv'


,Mes,Volumen total llegadas aeropuerto,Volumen llegadas Nacionales,Volumen llegadas Internacionales,Volumen llegadas Francia,Volumen llegadas Italia,Volumen llegadas Reino Unido,Volumen llegadas Países Bajos,Volumen llegadas Alemania,Volumen llegadas Resto del Mundo
0,2016-01,137801,78818,58983,15055,16652,11792,4790,1019,9675
1,2016-02,157979,89294,68685,19386,16888,14530,4953,1268,11660
2,2016-03,194938,108663,86275,25423,19613,16472,5978,3642,15147
3,2016-04,214451,114050,100401,35009,19197,15750,8215,5992,16238
4,2016-05,217285,110847,106438,33517,19969,16543,8908,5888,21613
5,2016-06,201755,106613,95142,29752,20048,15996,7638,5303,16405
6,2016-07,208472,104113,104359,34036,21080,17282,7777,6671,17513
7,2016-08,207707,108148,99559,28136,21841,17876,7036,6913,17757
8,2016-09,211285,106565,104720,31263,20647,16279,8756,7282,20493
9,2016-10,211995,103149,108846,33797,20412,16864,9492,7356,20925
